# Polynomial And Lasso Regression

Polynomial regression, Lasso, feature scaling, validation splits and optimization exercises.

Stage: learning experiment.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Lasso
from sklearn.model_selection import LeavePOut, cross_val_score
from sklearn.linear_model import LassoCV
from sklearn.pipeline import make_pipeline
import warnings
# Step 1: Read data from the assigned CSV file
data = pd.read_csv("rollnumber_76.csv", header=None)
data.columns = ['y', 'x']
plt.scatter(data['x'], data['y'])
plt.xlabel('x')
plt.ylabel('y')
plt.title('Scatter plot of y vs x')
plt.show()


In [ ]:
# Step 3: Split the data into training and testing sets (80:20 proportion)
X_train, X_test, y_train, y_test = train_test_split(data[['x']], data['y'],
test_size=0.2, random_state=42)
# Step 4: Tabulate the errors for different polynomial degrees (Table 1)
errors_table1 = []
degrees_table1 = list(range(1, 51))
for degree in degrees_table1:
    # Create polynomial features
    poly_features = PolynomialFeatures(degree=degree)
    X_train_poly = poly_features.fit_transform(X_train)
    X_test_poly = poly_features.transform(X_test)
    # Fit a linear regression model
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    # Predict on test data
    y_pred = model.predict(X_test_poly)
    # Calculate mean squared error
    error = mean_squared_error(y_test, y_pred)
    errors_table1.append(error)

# Step 5: Plot a graph with error on the y-axis and the degree of the
#polynomial (p) on the x-axis (Table 1)
plt.plot(degrees_table1, errors_table1, marker='o')
plt.xlabel('Degree of polynomial (p)')
plt.ylabel('Error')
plt.title('Error vs Degree of Polynomial (Table 1)')
plt.show()

In [ ]:
# Step 6: Tabulate the errors for different polynomial degrees (Table 1)
print("Degree of the polynomial (p)\tError")
for degree, error in zip(degrees_table1, errors_table1):
    print(f"{degree}\t\t\t\t{error:.6f}"  )

In [ ]:
# Step 7: Fit polynomial models of different degrees (up to 50) to the
#training data (Table 2)
errors_table2 = []
degrees_table2 = list(range(1, 51))
for degree in degrees_table2:
    # Create polynomial features
    poly_features = PolynomialFeatures(degree=degree)
    X_train_poly = poly_features.fit_transform(X_train)
    # Fit a LASSO regression model
    model = Lasso(alpha=0.01) # You can adjust the regularization strength(alpha) as needed
    model.fit(X_train_poly, y_train)
    # Predict on test data
    X_test_poly = poly_features.transform(X_test)
    y_pred = model.predict(X_test_poly)
    # Calculate mean squared error
    error = mean_squared_error(y_test, y_pred)
    errors_table2.append(error)

# Step 8: Plot a graph with error on the y-axis and the degree of the
#polynomial (p) on the x-axis (Table 2)

plt.plot(degrees_table2, errors_table2, marker='o')
plt.xlabel('Degree of polynomial (p)')
plt.ylabel('Error')
plt.title('Error vs Degree of Polynomial (Table 2)')
plt.show()
# Step 9: Tabulate the errors for different polynomial degrees (Table 2)
print("Degree of the polynomial (p)\tError")
for degree, error in zip(degrees_table2, errors_table2):
    print(f"{degree}\t\t\t\t{error:.6f}")

In [ ]:
# Polynomial model equations
def polynomial_model(degree):
    equation = f'y = '
    for i in range(degree+1):
        equation += f'β{i}x^{degree-i} + '
    return equation[:-3]  # Remove the last ' + '

# (c) Apt model from Table-1
apt_model_degree_table1 = degrees_table1[errors_table1.index(min(errors_table1))]
print(f"Apt model from Table-1: Degree {apt_model_degree_table1}")
print(f"Polynomial model from Table-1:{polynomial_model(apt_model_degree_table1)}")

In [ ]:
apt_model_degree_table2 = None
apt_model_degree_table2 = degrees_table2[errors_table2.index(min(errors_table2))]
print(f"Apt model from Table-2: Degree {apt_model_degree_table2}")
print(f"Polynomial model from Table-2:\n{polynomial_model(apt_model_degree_table2)}")

In [ ]:
# (e) Comparison of accuracies
mse_table1 = min(errors_table1)
mse_table2 = min(errors_table2)
if mse_table1 < mse_table2:
    print("The model obtained from Table-1 is more accurate. Linear modelperforms better.")
elif mse_table1 > mse_table2:
    print("The model obtained from Table-2 is more accurate.LASSO modelperforms better.")
else:
    print("Both models have similar accuracy.")
# Apt model equations
apt_model_table1 = polynomial_model(apt_model_degree_table1)
apt_model_table2 = polynomial_model(apt_model_degree_table2)
print("Mean squared error from Table-1:", mse_table1)
print("Mean squared error from Table-2:", mse_table2)
print(f"Polynomial model from Table-1: {apt_model_table1}")
print(f"Polynomial model from Table-2: {apt_model_table2}")

In [ ]:
# Degree of the polynomial
degree = apt_model_degree_table1
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)
# Extracting x and y from training data
x_train = train_data[['x']]
y_train = train_data['y']
# Creating polynomial features
polynomial_features = PolynomialFeatures(degree=degree)
x_poly_train = polynomial_features.fit_transform(x_train)
# Fitting linear regression model
model = LinearRegression()
model.fit(x_poly_train, y_train)
# Extracting coefficients
coefficients = model.coef_
# Extracting the intercept term from the linear regression model
intercept = model.intercept_
# Print the polynomial equation with the intercept term
print("Polynomial equation:")
equation = f"y = {intercept:.2f}"
for i in range(1, degree+1):
    equation += f" + {coefficients[i]:.2f} * x^{i}"
print(equation)
# Print the coefficients
print("Coefficients (β values):")
for i, coef in enumerate(coefficients[1:], start=1):
    print(f"β{i} = {coef:.2f}")

In [ ]:
degree = apt_model_degree_table2
# Regularization parameter (alpha) for Lasso
alpha = 0.01 # You may adjust this according to your findings
# Extracting x and y from training data
x_train = train_data[['x']]
y_train = train_data['y']
# Creating polynomial features
polynomial_features = PolynomialFeatures(degree=degree)
x_poly_train = polynomial_features.fit_transform(x_train)
# Fitting Lasso regression model
model = Lasso(alpha=alpha)
model.fit(x_poly_train, y_train)
# Extracting coefficients
coefficients = model.coef_
# Extracting the intercept term from the linear regression model
intercept = model.intercept_
# Print the polynomial equation with the intercept term
print("Polynomial equation:")
equation = f"y = {intercept:.2f}"
for i in range(1, degree+1):
    equation += f" + {coefficients[i]:.2f} * x^{i}"
print(equation)
# Print the coefficients
print("Coefficients (β values):")
for i, coef in enumerate(coefficients[1:], start=1):
    print(f"β{i} = {coef:.2f}")


In [ ]:
from sklearn.preprocessing import StandardScaler
# Standardize the features
scaler = StandardScaler()
X_train_poly_scaled = scaler.fit_transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)
best_degree = None
best_alpha = None
best_mse = float('inf')
# Iterate over different degrees and alpha values
for degree in range(1, 51):
    for alpha in [0.001, 0.01, 0.1, 1.0, 10.0]: # You can adjust the alphavalues
        # Create polynomial features for both training and test data
        poly_features = PolynomialFeatures(degree=degree, include_bias=False)
        X_train_poly = poly_features.fit_transform(train_data[['x']])
        X_test_poly = poly_features.transform(test_data[['x']])
        # Standardize the features
        scaler = StandardScaler()
        X_train_poly_scaled = scaler.fit_transform(X_train_poly)
        X_test_poly_scaled = scaler.transform(X_test_poly)
        # Fit LASSO regression model
        lasso_model = Lasso(alpha=alpha, max_iter=10000)
        lasso_model.fit(X_train_poly_scaled, train_data['y'])
        # Predict on the test data
        y_pred_lasso = lasso_model.predict(X_test_poly_scaled)
        # Calculate mean squared error
        mse_lasso = mean_squared_error(test_data['y'], y_pred_lasso)
        # Check if this combination gives a lower MSE
        if mse_lasso < best_mse:
            best_degree = degree
            best_alpha = alpha
            best_mse = mse_lasso
# Print the best results
print("Best Degree:", best_degree)
print("Best Alpha:", best_alpha)
print("Best Mean Squared Error (MSE):", best_mse)

In [ ]:
from sklearn.linear_model import LassoCV
# Define the degree of the polynomial
degree = apt_model_degree_table2
# Extracting x and y from the data
x_data = data[['x']]
y_data = data['y']
# Create polynomial features
poly_features = PolynomialFeatures(degree=degree, include_bias=False)
x_poly = poly_features.fit_transform(x_data)
# Initialize LASSO model with cross-validation
lasso_cv_model = LassoCV(alphas=np.logspace(-4, 4, 100), cv=10,
max_iter=10000) # 10-fold cross-validation
lasso_cv_model.fit(x_poly, y_data)
# Extract the coefficients from the LASSO model
lasso_coefficients = lasso_cv_model.coef_
lasso_intercept = lasso_cv_model.intercept_
# Print the sparse polynomial equation
print("Sparse Polynomial equation:")
equation_sparse = f"y = {lasso_intercept:.2f}"
for i, coef in enumerate(lasso_coefficients, start=1):
    equation_sparse += f" + {coef:.2f} * x^{i}"
print(equation_sparse)

In [ ]:
# LASSO model from Table-2
degree_table2 = apt_model_degree_table2 # Assuming this is the optimal degreefrom Table-2
poly_features_table2 = PolynomialFeatures(degree=degree_table2,
include_bias=False)
x_poly_train_table2 = poly_features_table2.fit_transform(X_train)
optimal_alpha = lasso_cv_model.alpha_
lasso_model_table2 = Lasso(alpha=optimal_alpha, max_iter=10000) # Increasemax_iter
lasso_model_table2.fit(x_poly_train_table2, y_train)
# LASSO model from Leave-p-out cross-validation
degree_leave_p_out = apt_model_degree_table2 # Assuming this is the optimaldegree from Leave-p-out cross-validation
poly_features_leave_p_out = PolynomialFeatures(degree=degree_leave_p_out,
include_bias=False)
x_poly_leave_p_out = poly_features_leave_p_out.fit_transform(x_data)
lasso_cv_model = LassoCV(alphas=np.logspace(-4, 4, 100), cv=10,
max_iter=10000) # Increase max_iter
lasso_cv_model = LassoCV()
lasso_cv_model.fit(x_poly_leave_p_out, y_data)

# Predictions
y_pred_table2 = lasso_model_table2.predict(poly_features_table2.transform(X_test))
y_pred_leave_p_out = lasso_cv_model.predict(x_poly_leave_p_out)
# Calculate mean squared errors
mse_table2 = mean_squared_error(y_test, y_pred_table2)
mse_leave_p_out = mean_squared_error(y_data, y_pred_leave_p_out)
# Print mean squared errors
print(f"MSE from Table-2 (LASSO): {mse_table2:.5f}")
print(f"MSE from Leave-p-out cross-validation (LASSO): {mse_leave_p_out:.5f}")
# Compare and print which model is more accurate
if mse_table2 < mse_leave_p_out:
    print("The LASSO model from Table-2 is more accurate.")
elif mse_table2 > mse_leave_p_out:
    print("The LASSO model from Leave-p-out cross-validation is moreaccurate.")
else:
    print("The two models have similar accuracy.")

In [ ]:
import numpy as np
r = np.log(21) / 2
A = np.array([[r + 0.1, 0], [0, r - 0.1]])
initial_condition = np.array([r - 5, r + 5])
alphas = [0.1, 0.2, 0.3, 0.4, 0.5] # List of step sizes
# Define the objective function and its gradient
def f(x):
    return (x - (r * 0.1)).T @ A @ (x - (r * 0.1))
def gradient_f(x):
    return 2 * A @ (x - (r * 0.1))
# Display header for the table
print(f"{'Iteration':<10}{'δ':<5}{'x':<30}{'f(x)':<30}{'Gradient':<30}")
# Perform gradient descent iterations for each step size
for alpha in alphas:
      # Re-initialize variables for each step size
      x = initial_condition
      iterations=30
      # Perform gradient descent iterations
      for iteration in range(iterations):
          # Calculate values for the current iteration
          fx = f(x)
          gradient = gradient_f(x)
          # Display values in the table
          print(f"{iteration:<10}{alpha:<5}{np.array2string(x, precision=5,separator=','):<30}{fx:<30}{np.array2string(gradient, precision=5,separator=','):<30}")
          # Update x using gradient descent
          x = x - alpha * gradient
          # Check for convergence (you can use a different convergencecriterion)
          if np.linalg.norm(gradient) < 1e-6:
              print("\nConverged!")
              break
      print("\n" + "="*80 + "\n") # Add a separator between different stepsizes